# Original PartImageNet Black-Mask Occlusion: v39 and ResNet

This notebook follows the protocol of `partimagenet_original_black_mask_occlusion_v14_resnet.ipynb`: controlled occlusion with centered black square masks on original PartImageNet validation images.

Mask area fractions are `0.36`, `0.49`, and `0.64` of the image area. Because the mask is square, those correspond to side ratios `0.60`, `0.70`, and `0.80` of the 224x224 analysis canvas.

The three models classify the clean and masked images:

- `logits`: Strict AOG v39 skeleton-primary-relation model
- `resnet_pretrained`: ResNet-50 initialized from ImageNet weights, then fine-tuned on PartImageNet
- `resnet_scratch`: ResNet-50 trained from random initialization on PartImageNet

The image selection is reused from the original v14 black-mask experiment so the occlusion readout is directly comparable at the protocol level.


In [1]:
from pathlib import Path
import importlib.util
import json
import random
import sys
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageDraw
from IPython.display import display, Markdown
from tqdm.auto import tqdm

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists() and (candidate / 'configs' / 'stage1_quality_upgrade.yaml').exists():
            return candidate
    return start


REPO_ROOT = find_repo_root()
EXPERIMENT_ROOT = REPO_ROOT / 'experiments' / 'aog_v19_v23'
for module_path in [REPO_ROOT / 'src', EXPERIMENT_ROOT / 'src']:
    if str(module_path) not in sys.path:
        sys.path.insert(0, str(module_path))

CONFIG = Path('configs/stage1_quality_upgrade.yaml')
PART_ROOT = Path('../full_hyco/PartImageNet')
STAGE1_CKPT = Path('runs/stage1_quality_upgrade/checkpoints/stage1_best.pt')
CALIBRATION_PATH = Path('runs/strict_aog_cache_v17/terminal_calibration.json')
STRICT_GRAMMAR = Path('experiments/aog_v19_v23/runs/strict_aog_cache_v39/strict_aog_v39.pt')
STRICT_CKPT = Path('experiments/aog_v19_v23/runs/strict_aog_v39_skeleton_primary_relation_templates/checkpoints/strict_aog_best.pt')
RESNET_PRETRAINED_CKPT = Path('runs/resnet50_partimagenet_v2/checkpoints/resnet_best.pt')
RESNET_SCRATCH_CKPT = Path('runs/resnet50_partimagenet_v2_scratch/checkpoints/resnet_best.pt')
RESNET_MODEL = 'resnet50'
RESNET_PREPROCESS = 'imagenet'
RESNET_IMG_SIZE = 224
RESNET_VAL_RESIZE = 256

OUT = Path('experiments/aog_v19_v23/runs/partimagenet_original_black_mask_occlusion_v39_resnet')
PREVIOUS_SELECTION = Path('runs/partimagenet_original_occlusion_v14_resnet/selected_original_occlusion_cases.csv')

CONFIG = CONFIG if CONFIG.is_absolute() else REPO_ROOT / CONFIG
PART_ROOT = PART_ROOT if PART_ROOT.is_absolute() else (REPO_ROOT / PART_ROOT).resolve()
STAGE1_CKPT = STAGE1_CKPT if STAGE1_CKPT.is_absolute() else REPO_ROOT / STAGE1_CKPT
CALIBRATION_PATH = CALIBRATION_PATH if CALIBRATION_PATH.is_absolute() else REPO_ROOT / CALIBRATION_PATH
STRICT_GRAMMAR = STRICT_GRAMMAR if STRICT_GRAMMAR.is_absolute() else REPO_ROOT / STRICT_GRAMMAR
STRICT_CKPT = STRICT_CKPT if STRICT_CKPT.is_absolute() else REPO_ROOT / STRICT_CKPT
RESNET_PRETRAINED_CKPT = RESNET_PRETRAINED_CKPT if RESNET_PRETRAINED_CKPT.is_absolute() else REPO_ROOT / RESNET_PRETRAINED_CKPT
RESNET_SCRATCH_CKPT = RESNET_SCRATCH_CKPT if RESNET_SCRATCH_CKPT.is_absolute() else REPO_ROOT / RESNET_SCRATCH_CKPT
OUT = OUT if OUT.is_absolute() else REPO_ROOT / OUT
PREVIOUS_SELECTION = PREVIOUS_SELECTION if PREVIOUS_SELECTION.is_absolute() else REPO_ROOT / PREVIOUS_SELECTION
OUT.mkdir(parents=True, exist_ok=True)

ANALYSIS_IMAGE_SIZE = 224
MASK_AREA_FRACTIONS = [0.36, 0.49, 0.64]
MASK_COLOR = (0, 0, 0)
EVAL_BATCH_SIZE = 16
IMAGES_PER_CLASS = 2
CANDIDATES_PER_CLASS = 24
DEVICE = 'auto'
SEED = 123

for required in [CONFIG, STAGE1_CKPT, CALIBRATION_PATH, STRICT_GRAMMAR, STRICT_CKPT, RESNET_PRETRAINED_CKPT, RESNET_SCRATCH_CKPT]:
    p = required if required.is_absolute() else REPO_ROOT / required
    if not p.exists():
        raise FileNotFoundError(p)

helper_path = REPO_ROOT / 'scripts' / 'partimagenet_gatys_cue_conflict.py'
spec = importlib.util.spec_from_file_location('gatys_helper', helper_path)
gatys = importlib.util.module_from_spec(spec)
assert spec.loader is not None
sys.modules[spec.name] = gatys
spec.loader.exec_module(gatys)

resnet_helper_path = REPO_ROOT / 'scripts' / 'compare_resnet_partimagenet_gatys_cue_conflict.py'
resnet_spec = importlib.util.spec_from_file_location('resnet_compare_helper', resnet_helper_path)
resnet_cmp = importlib.util.module_from_spec(resnet_spec)
assert resnet_spec.loader is not None
sys.modules[resnet_spec.name] = resnet_cmp
resnet_spec.loader.exec_module(resnet_cmp)

from partcat_hkg.config import load_config
from partcat_hkg.data.loaders import make_datasets
from partcat_hkg.strict_aog.grammar import load_strict_aog
from partcat_hkg.strict_aog.parser import ParserConfig, StrictAOGParser
from partcat_hkg.strict_aog.terminals import TerminalExtractionConfig, batch_extract_terminals, load_terminal_calibration
from partcat_hkg.utils.seed import set_seed

BRANCHES = ['logits', 'resnet_pretrained', 'resnet_scratch']
BRANCH_LABELS = {
    'logits': 'Strict AOG v39 skeleton primary relations',
    'resnet_pretrained': 'ResNet-50 pretrained+FT',
    'resnet_scratch': 'ResNet-50 scratch',
}
COLORS = {
    'logits': '#3b82f6',
    'resnet_pretrained': '#f59e0b',
    'resnet_scratch': '#7c3aed',
}

pd.set_option('display.max_columns', 140)
pd.set_option('display.width', 180)
print('output:', OUT)
print('mask area fractions:', MASK_AREA_FRACTIONS)

output: /home/dfli/instance_slot_aog/experiments/aog_v19_v23/runs/partimagenet_original_black_mask_occlusion_v39_resnet
mask area fractions: [0.36, 0.49, 0.64]


/home/dfli/anaconda3/envs/partviz/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Data and Models

In [2]:
PARSER_CFG = ParserConfig(
    assignment='gpu_mf',
    beam_size=8,
    top_terminals_per_slot=4,
    class_chunk=0,
    mf_iters=4,
    mf_tau=0.35,
    mf_column_iters=8,
    mf_edge_chunk_size=128,
    relation_weight=0.20,
    role_overlap_weight=0.40,
    min_role_overlap=0.0,
    min_parse_role_overlap=0.25,
    low_role_penalty=0.90,
    min_parse_inst_edges=2.0,
    low_inst_edge_penalty=0.75,
    min_parse_edge_coverage=0.40,
    low_edge_coverage_penalty=0.75,
    count_weight=0.20,
    count_role_power=0.5,
    count_source='assigned',
    count_model='categorical',
    count_score_mode='raw',
    count_positive_scale=1.00,
    count_negative_scale=1.00,
    reason_weight=0.00,
    role_overlap_floor=0.02,
    count_ll_clip_min=-8.0,
    count_ll_clip_max=4.0,
    edge_missing_weight=0.75,
    edge_score_mode='peer_llr',
    edge_background_min_count=8.0,
    class_prior_weight=0.0,
    edge_info_gain_power=0.5,
    edge_gate_floor=0.30,
    typed_edge_weight=1.00,
    semantic_edge_weight=1.35,
    edge_positive_scale=0.35,
    edge_negative_scale=1.00,
    score_normalization='',
    node_score_normalization='none',
    edge_score_normalization='sqrt',
    node_app_weight=0.30,
    node_geom_weight=0.35,
    node_presence_weight=0.05,
    slot_prior_weight=0.02,
    missing_weight=0.45,
    spurious_weight=0.05,
    template_tau=0.75,
)

cfg = load_config(str(CONFIG))
if PART_ROOT:
    cfg.paths.partimagenet_root = str(PART_ROOT)
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
device = gatys.resolve_device(DEVICE)
train_ds, val_ds = make_datasets(cfg)
class_names = list(train_ds.schema.obj_names)
part_names = list(train_ds.schema.part_names)
calibration = load_terminal_calibration(CALIBRATION_PATH, part_names=part_names)
TERMINAL_CFG = TerminalExtractionConfig(
    threshold=0.40,
    min_area_frac=1e-4,
    min_presence=0.05,
    max_components_per_part=4,
    max_terminals=32,
    mask_size=64,
    **calibration,
)
stage_transform = gatys.ImageOnlyTransform(cfg.data.img_size, train=False)
resnet_transform = resnet_cmp.make_resnet_eval_transform(RESNET_PREPROCESS, RESNET_IMG_SIZE, RESNET_VAL_RESIZE)

stage1 = gatys.load_stage1_model(SimpleNamespace(stage1_ckpt=str(STAGE1_CKPT), allow_partial_stage1_load=False), cfg, train_ds.schema, device)
grammar = load_strict_aog(str(STRICT_GRAMMAR))
strict_model = StrictAOGParser(grammar, PARSER_CFG).to(device)
payload = torch.load(str(STRICT_CKPT), map_location='cpu')
state = payload.get('model', payload.get('state_dict', payload)) if isinstance(payload, dict) else payload
strict_model.load_state_dict(state, strict=True)
strict_model.eval()

def load_resnet_baseline(branch: str, ckpt: Path) -> torch.nn.Module:
    model = resnet_cmp.make_resnet(RESNET_MODEL, len(class_names), imagenet_pretrained=False).to(device)
    extra = resnet_cmp.load_resnet_checkpoint(ckpt, model, strict=True)
    if extra.get('class_names'):
        print(f'{branch} classes:', extra['class_names'])
    model.eval()
    return model

resnet_models = {
    'resnet_pretrained': load_resnet_baseline('resnet_pretrained', RESNET_PRETRAINED_CKPT),
    'resnet_scratch': load_resnet_baseline('resnet_scratch', RESNET_SCRATCH_CKPT),
}

model_meta = pd.DataFrame([
    {'branch_key': 'logits', 'branch': BRANCH_LABELS['logits'], 'checkpoint': str(STRICT_CKPT), 'grammar': str(STRICT_GRAMMAR)},
    {'branch_key': 'resnet_pretrained', 'branch': BRANCH_LABELS['resnet_pretrained'], 'checkpoint': str(RESNET_PRETRAINED_CKPT)},
    {'branch_key': 'resnet_scratch', 'branch': BRANCH_LABELS['resnet_scratch'], 'checkpoint': str(RESNET_SCRATCH_CKPT)},
])
display(model_meta)
print('val samples:', len(val_ds), 'classes:', class_names)

[dataset] train.json: 20457 samples | C=11 F=13 R=40
[dataset] val.json: 1205 samples | C=11 F=13 R=40
[datasets] train: /home/dfli/full_hyco/PartImageNet/annotations/train/train.json | images: /home/dfli/full_hyco/PartImageNet/images/train | samples: 20457
[datasets] val: /home/dfli/full_hyco/PartImageNet/annotations/val/val.json | images: /home/dfli/full_hyco/PartImageNet/images/val | samples: 1205


/home/dfli/anaconda3/envs/partviz/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/dfli/anaconda3/envs/partviz/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened becau

resnet_pretrained classes: ['aeroplane', 'bicycle', 'biped', 'bird', 'boat', 'bottle', 'car', 'fish', 'quadruped', 'reptile', 'snake']


resnet_scratch classes: ['aeroplane', 'bicycle', 'biped', 'bird', 'boat', 'bottle', 'car', 'fish', 'quadruped', 'reptile', 'snake']


,branch_key,branch,checkpoint,grammar
0,logits,Strict AOG v39 skeleton primary relations,/home/dfli/instance_slot_aog/experiments/aog_v...,/home/dfli/instance_slot_aog/experiments/aog_v...
1,resnet_pretrained,ResNet-50 pretrained+FT,/home/dfli/instance_slot_aog/runs/resnet50_par...,NaN
2,resnet_scratch,ResNet-50 scratch,/home/dfli/instance_slot_aog/runs/resnet50_par...,NaN


val samples: 1205 classes: ['aeroplane', 'bicycle', 'biped', 'bird', 'boat', 'bottle', 'car', 'fish', 'quadruped', 'reptile', 'snake']


## Strict AOG Class Templates

This chunk visualizes the learned v39 AOG templates before the occlusion run. Each row is one object class and each column is one And-template alternative. Slots are drawn at their learned normalized geometry and labeled by part role; relation edges are drawn from the grammar between slots.

In [3]:
from collections import Counter, defaultdict
from matplotlib.patches import Rectangle

EDGE_TYPE_NAMES = {
    0: 'anchor',
    1: 'repeat',
    2: 'generic',
    3: 'structural',
    4: 'semantic',
    5: 'ATTACH_BOND',
    6: 'HINGE',
    7: 'BUTTING',
    8: 'CONCENTRIC',
    9: 'BAR_CIRCLE',
    10: 'AXIAL_ALIGN',
    11: 'BILATERAL_SYMMETRY',
    12: 'CONTAINS',
    13: 'SUPPORTS',
    14: 'OCCLUDES',
}


def template_summary_tables(grammar):
    template_rows = []
    edge_rows = []
    edges = grammar.edges.detach().cpu().long()
    edge_types = grammar.edge_type.detach().cpu().long()
    edge_support = grammar.edge_support.detach().cpu().float()
    edge_info_gain = grammar.edge_info_gain.detach().cpu().float()
    for c, cls_name in enumerate(grammar.class_names):
        for a in range(int(grammar.num_templates)):
            if float(grammar.template_valid[c, a].item()) <= 0.5:
                continue
            slot_parts = []
            for s in range(int(grammar.max_slots)):
                if float(grammar.slot_valid[c, a, s].item()) <= 0.5:
                    continue
                part_id = int(grammar.slot_part[c, a, s].item())
                part_name = grammar.part_names[part_id] if 0 <= part_id < len(grammar.part_names) else str(part_id)
                presence = float(grammar.slot_presence[c, a, s].item())
                cx, cy, w, h, area, score = [float(x) for x in grammar.slot_geom_mean[c, a, s].detach().cpu().tolist()]
                slot_parts.append(part_name)
            edge_idx = ((edges[:, 0] == c) & (edges[:, 1] == a)).nonzero(as_tuple=False).flatten().tolist() if edges.numel() else []
            type_counts = Counter(int(edge_types[i].item()) for i in edge_idx)
            template_rows.append({
                'class_id': c,
                'class_name': cls_name,
                'template_id': a,
                'template_prior': float(grammar.template_prior[c, a].item()),
                'num_slots': len(slot_parts),
                'num_edges': len(edge_idx),
                'slot_parts': ', '.join(slot_parts),
                'edge_types': ', '.join(f"{EDGE_TYPE_NAMES.get(t, str(t))}:{n}" for t, n in sorted(type_counts.items())),
            })
            for i in edge_idx:
                _, _, si, sj = [int(x) for x in edges[i].tolist()]
                pi = int(grammar.slot_part[c, a, si].item())
                pj = int(grammar.slot_part[c, a, sj].item())
                edge_rows.append({
                    'class_id': c,
                    'class_name': cls_name,
                    'template_id': a,
                    'slot_i': si,
                    'slot_j': sj,
                    'part_i': grammar.part_names[pi] if 0 <= pi < len(grammar.part_names) else str(pi),
                    'part_j': grammar.part_names[pj] if 0 <= pj < len(grammar.part_names) else str(pj),
                    'edge_type': int(edge_types[i].item()),
                    'edge_type_name': EDGE_TYPE_NAMES.get(int(edge_types[i].item()), str(int(edge_types[i].item()))),
                    'edge_support': float(edge_support[i].item()),
                    'edge_info_gain': float(edge_info_gain[i].item()),
                })
    return pd.DataFrame(template_rows), pd.DataFrame(edge_rows)


template_table, template_edge_table = template_summary_tables(grammar)
template_table.to_csv(OUT / 'strict_aog_v39_class_template_summary.csv', index=False)
template_edge_table.to_csv(OUT / 'strict_aog_v39_class_template_edges.csv', index=False)
display(template_table)


def plot_class_templates(grammar, *, out_path: Path):
    num_classes = int(grammar.num_classes)
    num_templates = int(grammar.num_templates)
    fig, axes = plt.subplots(
        num_classes,
        num_templates,
        figsize=(4.8 * num_templates, 2.8 * num_classes),
        constrained_layout=True,
    )
    if num_classes == 1:
        axes = np.array([axes])
    if num_templates == 1:
        axes = axes[:, None]
    edges = grammar.edges.detach().cpu().long()
    edge_types = grammar.edge_type.detach().cpu().long()
    palette = list(plt.cm.tab20(np.linspace(0, 1, 20)))

    for c, cls_name in enumerate(grammar.class_names):
        for a in range(num_templates):
            ax = axes[c, a]
            ax.set_xlim(0.0, 1.0)
            ax.set_ylim(1.0, 0.0)
            ax.set_aspect('equal', adjustable='box')
            ax.set_xticks([])
            ax.set_yticks([])
            if a == 0:
                ax.set_ylabel(cls_name, fontsize=10, rotation=0, ha='right', va='center')
            if float(grammar.template_valid[c, a].item()) <= 0.5:
                ax.set_title(f'template {a}: invalid', fontsize=9)
                ax.set_facecolor('#f4f4f5')
                continue
            ax.set_title(f"T{a} prior={float(grammar.template_prior[c, a].item()):.2f}", fontsize=9)
            valid_slots = []
            for s in range(int(grammar.max_slots)):
                if float(grammar.slot_valid[c, a, s].item()) > 0.5:
                    valid_slots.append(s)
            slot_pos = {}
            for s in valid_slots:
                cx, cy, w, h, area, score = [float(x) for x in grammar.slot_geom_mean[c, a, s].detach().cpu().tolist()]
                cx = min(max(cx, 0.03), 0.97)
                cy = min(max(cy, 0.03), 0.97)
                slot_pos[s] = (cx, cy)

            edge_idx = ((edges[:, 0] == c) & (edges[:, 1] == a)).nonzero(as_tuple=False).flatten().tolist() if edges.numel() else []
            pair_types = defaultdict(list)
            for i in edge_idx:
                _, _, si, sj = [int(x) for x in edges[i].tolist()]
                if si in slot_pos and sj in slot_pos:
                    key = tuple(sorted((si, sj)))
                    pair_types[key].append(EDGE_TYPE_NAMES.get(int(edge_types[i].item()), str(int(edge_types[i].item()))))
            for (si, sj), types in pair_types.items():
                x0, y0 = slot_pos[si]
                x1, y1 = slot_pos[sj]
                ax.plot([x0, x1], [y0, y1], color='#64748b', alpha=0.28, linewidth=0.7 + 0.25 * min(len(types), 6), zorder=1)

            for s in valid_slots:
                part_id = int(grammar.slot_part[c, a, s].item())
                part_name = grammar.part_names[part_id] if 0 <= part_id < len(grammar.part_names) else str(part_id)
                cx, cy, w, h, area, score = [float(x) for x in grammar.slot_geom_mean[c, a, s].detach().cpu().tolist()]
                w = min(max(w, 0.035), 0.28)
                h = min(max(h, 0.035), 0.28)
                cx = min(max(cx, 0.03), 0.97)
                cy = min(max(cy, 0.03), 0.97)
                color = palette[part_id % len(palette)]
                rect = Rectangle(
                    (cx - w / 2, cy - h / 2),
                    w,
                    h,
                    facecolor=color,
                    edgecolor='black',
                    linewidth=0.7,
                    alpha=0.38,
                    zorder=2,
                )
                ax.add_patch(rect)
                ax.scatter([cx], [cy], s=18, color=[color[:3]], edgecolors='black', linewidths=0.4, zorder=3)
                ax.text(
                    cx,
                    cy,
                    f"{s}:{part_name}",
                    fontsize=6.5,
                    ha='center',
                    va='center',
                    color='black',
                    bbox=dict(facecolor='white', alpha=0.72, edgecolor='none', pad=0.8),
                    zorder=4,
                )
            ax.text(
                0.01,
                0.99,
                f"slots={len(valid_slots)} edges={len(edge_idx)}",
                transform=ax.transAxes,
                ha='left',
                va='top',
                fontsize=7,
                color='#334155',
            )
    fig.savefig(out_path, dpi=180)
    plt.show()


plot_class_templates(grammar, out_path=OUT / 'strict_aog_v39_class_templates.png')
display(pd.DataFrame({'artifact': [
    str(OUT / 'strict_aog_v39_class_templates.png'),
    str(OUT / 'strict_aog_v39_class_template_summary.csv'),
    str(OUT / 'strict_aog_v39_class_template_edges.csv'),
]}))

,class_id,class_name,template_id,template_prior,num_slots,num_edges,slot_parts,edge_types
0,0,aeroplane,0,0.584270,9,12,"body, engine, engine, head, tail, wing, wing, ...","ATTACH_BOND:10, BILATERAL_SYMMETRY:2"
1,0,aeroplane,1,0.318352,6,8,"body, engine, engine, head, tail, wing",ATTACH_BOND:8
2,0,aeroplane,2,0.097378,10,13,"body, engine, engine, engine, engine, head, ta...",ATTACH_BOND:13
3,1,bicycle,0,0.561694,5,5,"body, head, seat, wheel, wheel","ATTACH_BOND:2, BAR_CIRCLE:2, BILATERAL_SYMMETRY:1"
4,1,bicycle,1,0.329650,4,2,"body, head, seat, wheel","ATTACH_BOND:1, BAR_CIRCLE:1"
5,1,bicycle,2,0.108656,6,7,"body, head, seat, wheel, wheel, wheel","BAR_CIRCLE:6, BILATERAL_SYMMETRY:1"
6,2,biped,0,0.392358,8,9,"body, foot, foot, hand, hand, hand, head, tail","ATTACH_BOND:6, HINGE:2, SUPPORTS:1"
7,2,biped,1,0.360671,6,8,"body, foot, foot, hand, head, tail","ATTACH_BOND:5, HINGE:1, BUTTING:1, SUPPORTS:1"
8,2,biped,2,0.246971,3,3,"body, hand, head","ATTACH_BOND:1, BUTTING:1, SUPPORTS:1"
9,3,bird,0,0.493976,6,6,"body, foot, head, tail, wing, wing","ATTACH_BOND:4, AXIAL_ALIGN:2"


/tmp/codex-tmp/ipykernel_5069/816233310.py:179: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,artifact
0,/home/dfli/instance_slot_aog/experiments/aog_v...
1,/home/dfli/instance_slot_aog/experiments/aog_v...
2,/home/dfli/instance_slot_aog/experiments/aog_v...


## Select Original Images

This reuses the 22-image original occlusion selection from `runs/partimagenet_original_occlusion_v14_resnet`, matching the v14 black-mask notebook's data-selection protocol. The selection contains two validation images per class that were clean-correct for the original v14/ResNet experiment; the clean row in the table below reports how v39 behaves on the same fixed selection.

In [4]:
def original_canvas(path: str | Path, size: int = ANALYSIS_IMAGE_SIZE) -> Image.Image:
    p = Path(path)
    if not p.is_absolute():
        p = REPO_ROOT / p
    img = Image.open(p).convert('RGB')
    return ImageOps.fit(img, (size, size), method=Image.Resampling.BICUBIC, centering=(0.5, 0.5))


def apply_stage_transform(img: Image.Image) -> torch.Tensor:
    x, _raw = stage_transform(img)
    return x


def apply_resnet_transform(img: Image.Image) -> torch.Tensor:
    return resnet_cmp.apply_image_transform(resnet_transform, img)


def batched(items: list, batch_size: int):
    for i in range(0, len(items), int(batch_size)):
        yield items[i:i + int(batch_size)]


@torch.no_grad()
def predict_pils(images: list[Image.Image], *, batch_size: int = EVAL_BATCH_SIZE) -> dict[str, torch.Tensor]:
    out: dict[str, list[torch.Tensor]] = {branch: [] for branch in BRANCHES}
    for chunk in batched(images, batch_size):
        stage_images = torch.stack([apply_stage_transform(img) for img in chunk], dim=0).to(device, non_blocking=True)
        stage1_out = stage1(stage_images)
        terminals = batch_extract_terminals(stage1_out, cfg=TERMINAL_CFG)
        strict_out = strict_model(terminals, enable_edges=True)
        out['logits'].append(strict_out['logits'].detach().cpu())

        resnet_images = torch.stack([apply_resnet_transform(img) for img in chunk], dim=0).to(device, non_blocking=True)
        for branch, model in resnet_models.items():
            out[branch].append(model(resnet_images).detach().cpu())
    return {branch: torch.cat(parts, dim=0) for branch, parts in out.items()}


def target_scores(logits: torch.Tensor, target: int) -> dict:
    logits = logits.detach().cpu()
    target = int(target)
    pred = int(logits.argmax().item())
    others = logits.clone()
    others[target] = -torch.inf
    max_other_idx = int(others.argmax().item())
    max_other = float(others[max_other_idx].item())
    target_logit = float(logits[target].item())
    probs = torch.softmax(logits.float(), dim=0)
    return {
        'pred': pred,
        'pred_name': class_names[pred],
        'correct': int(pred == target),
        'target_logit': target_logit,
        'max_other_label': max_other_idx,
        'max_other_name': class_names[max_other_idx],
        'max_other_logit': max_other,
        'target_margin': target_logit - max_other,
        'target_prob': float(probs[target].item()),
    }


def build_selection_from_val() -> pd.DataFrame:
    groups: dict[int, list[dict]] = {}
    for idx, rec in enumerate(val_ds.samples):
        groups.setdefault(int(rec['obj_label']), []).append({'val_index': idx, **rec})
    candidate_rows = []
    for label, rows in sorted(groups.items()):
        rng = random.Random(SEED + int(label))
        rows = rows.copy()
        rng.shuffle(rows)
        for rec in rows[:CANDIDATES_PER_CLASS]:
            candidate_rows.append({
                'val_index': int(rec['val_index']),
                'image_path': str(rec['img_path']),
                'target_label': int(rec['obj_label']),
                'target_name': class_names[int(rec['obj_label'])],
            })
    images = [original_canvas(r['image_path']) for r in candidate_rows]
    logits_by_branch = predict_pils(images)
    for i, row in enumerate(candidate_rows):
        target = int(row['target_label'])
        for branch in BRANCHES:
            scores = target_scores(logits_by_branch[branch][i], target)
            for k, v in scores.items():
                row[f'{branch}_{k}'] = v
        row['all_three_correct'] = int(all(row[f'{b}_correct'] for b in BRANCHES))
        row['min_target_margin'] = min(float(row[f'{b}_target_margin']) for b in BRANCHES)
    candidate_df = pd.DataFrame(candidate_rows)
    parts = []
    for label, grp in candidate_df.groupby('target_label'):
        primary = grp[grp['all_three_correct'].eq(1)].sort_values('min_target_margin', ascending=False)
        fallback = grp.sort_values('min_target_margin', ascending=False)
        parts.append(pd.concat([primary, fallback]).drop_duplicates('val_index').head(IMAGES_PER_CLASS))
    return pd.concat(parts, ignore_index=True).sort_values(['target_label', 'val_index']).reset_index(drop=True)

if PREVIOUS_SELECTION.exists():
    selected = pd.read_csv(PREVIOUS_SELECTION)
    # Keep only the fields this experiment needs; the previous selection already
    # contains clean predictions for all three models.
    selected = selected.rename(columns={'target_label': 'target_label', 'target_name': 'target_name'}).copy()
    print('loaded previous selection:', PREVIOUS_SELECTION)
else:
    selected = build_selection_from_val()
    print('built fresh selection from validation data')

selected.to_csv(OUT / 'selected_black_mask_original_cases.csv', index=False)
display(selected[['val_index', 'target_name', 'image_path']].head(30))
print('selected images:', len(selected))

loaded previous selection: /home/dfli/instance_slot_aog/runs/partimagenet_original_occlusion_v14_resnet/selected_original_occlusion_cases.csv


,val_index,target_name,image_path
0,1159,aeroplane,../full_hyco/PartImageNet/images/val/n02690373...
1,1171,aeroplane,../full_hyco/PartImageNet/images/val/n04552348...
2,1179,bicycle,../full_hyco/PartImageNet/images/val/n03792782...
3,1201,bicycle,../full_hyco/PartImageNet/images/val/n03792782...
4,165,biped,../full_hyco/PartImageNet/images/val/n02492660...
5,179,biped,../full_hyco/PartImageNet/images/val/n02480495...
6,766,bird,../full_hyco/PartImageNet/images/val/n02002724...
7,778,bird,../full_hyco/PartImageNet/images/val/n02033041...
8,412,boat,../full_hyco/PartImageNet/images/val/n04483307...
9,413,boat,../full_hyco/PartImageNet/images/val/n04483307...


selected images: 22


## Apply Centered Black Masks and Classify

In [5]:
def add_black_mask(img: Image.Image, area_fraction: float) -> tuple[Image.Image, dict]:
    area_fraction = float(area_fraction)
    side_ratio = area_fraction ** 0.5
    side = int(round(ANALYSIS_IMAGE_SIZE * side_ratio))
    side = max(1, min(ANALYSIS_IMAGE_SIZE, side))
    x0 = (ANALYSIS_IMAGE_SIZE - side) // 2
    y0 = (ANALYSIS_IMAGE_SIZE - side) // 2
    x1 = x0 + side
    y1 = y0 + side
    out = img.copy()
    draw = ImageDraw.Draw(out)
    draw.rectangle([x0, y0, x1 - 1, y1 - 1], fill=MASK_COLOR)
    return out, {
        'mask_area_fraction': area_fraction,
        'mask_side_ratio': side_ratio,
        'mask_side_px': side,
        'mask_x0': x0,
        'mask_y0': y0,
        'mask_x1': x1,
        'mask_y1': y1,
    }

condition_rows = []
image_variants = []
for _, row in selected.iterrows():
    clean = original_canvas(row['image_path'])
    variants = [(0.0, clean, {'mask_area_fraction': 0.0, 'mask_side_ratio': 0.0, 'mask_side_px': 0, 'mask_x0': -1, 'mask_y0': -1, 'mask_x1': -1, 'mask_y1': -1})]
    for area in MASK_AREA_FRACTIONS:
        masked, mask_meta = add_black_mask(clean, area)
        variants.append((float(area), masked, mask_meta))
    for area, img, mask_meta in variants:
        image_variants.append(img)
        condition_rows.append({
            'val_index': int(row['val_index']),
            'image_path': row['image_path'],
            'target_label': int(row['target_label']),
            'target_name': row['target_name'],
            **mask_meta,
        })

logits_by_branch = predict_pils(image_variants)
records = []
for i, row in enumerate(condition_rows):
    target = int(row['target_label'])
    for branch in BRANCHES:
        scores = target_scores(logits_by_branch[branch][i], target)
        records.append({
            **row,
            'branch_key': branch,
            'branch': BRANCH_LABELS[branch],
            **scores,
        })

results = pd.DataFrame(records)
clean_lookup = results[results['mask_area_fraction'].eq(0.0)].set_index(['val_index', 'branch_key'])
results['clean_pred'] = [clean_lookup.loc[(r.val_index, r.branch_key), 'pred'] for r in results.itertuples()]
results['clean_correct'] = [clean_lookup.loc[(r.val_index, r.branch_key), 'correct'] for r in results.itertuples()]
results['clean_target_margin'] = [clean_lookup.loc[(r.val_index, r.branch_key), 'target_margin'] for r in results.itertuples()]
results['clean_target_prob'] = [clean_lookup.loc[(r.val_index, r.branch_key), 'target_prob'] for r in results.itertuples()]
results['target_margin_drop'] = results['clean_target_margin'] - results['target_margin']
results['target_prob_drop'] = results['clean_target_prob'] - results['target_prob']
results['top1_flip_from_clean'] = (results['pred'] != results['clean_pred']).astype(int)
results['correct_flip_from_clean'] = (results['correct'] != results['clean_correct']).astype(int)

results.to_csv(OUT / 'black_mask_classification_results.csv', index=False)
display(results.head())
print('rows:', len(results))

,val_index,image_path,target_label,target_name,mask_area_fraction,mask_side_ratio,mask_side_px,mask_x0,mask_y0,mask_x1,mask_y1,branch_key,branch,pred,pred_name,correct,target_logit,max_other_label,max_other_name,max_other_logit,target_margin,target_prob,clean_pred,clean_correct,clean_target_margin,clean_target_prob,target_margin_drop,target_prob_drop,top1_flip_from_clean,correct_flip_from_clean
0,1159,../full_hyco/PartImageNet/images/val/n02690373...,0,aeroplane,0.00,0.0,0,-1,-1,-1,-1,logits,Strict AOG v39 skeleton primary relations,0,aeroplane,1,2.667227,3,bird,-1.116844,3.784070,0.968707,0,1,3.784070,0.968707,0.000000,0.000000,0,0
1,1159,../full_hyco/PartImageNet/images/val/n02690373...,0,aeroplane,0.00,0.0,0,-1,-1,-1,-1,resnet_pretrained,ResNet-50 pretrained+FT,0,aeroplane,1,2.850697,8,quadruped,-2.409402,5.260099,0.956637,0,1,5.260099,0.956637,0.000000,0.000000,0,0
2,1159,../full_hyco/PartImageNet/images/val/n02690373...,0,aeroplane,0.00,0.0,0,-1,-1,-1,-1,resnet_scratch,ResNet-50 scratch,0,aeroplane,1,4.910744,6,car,-0.250443,5.161187,0.974941,0,1,5.161187,0.974941,0.000000,0.000000,0,0
3,1159,../full_hyco/PartImageNet/images/val/n02690373...,0,aeroplane,0.36,0.6,134,45,45,179,179,logits,Strict AOG v39 skeleton primary relations,5,bottle,0,-3.260120,5,bottle,-2.107645,-1.152476,0.129905,0,1,3.784070,0.968707,4.936546,0.838801,1,1
4,1159,../full_hyco/PartImageNet/images/val/n02690373...,0,aeroplane,0.36,0.6,134,45,45,179,179,resnet_pretrained,ResNet-50 pretrained+FT,0,aeroplane,1,-1.026181,3,bird,-3.745131,2.718950,0.910426,0,1,5.260099,0.956637,2.541149,0.046212,0,0


rows: 264


## Summary

In [6]:
summary = results.groupby(['branch_key', 'branch', 'mask_area_fraction', 'mask_side_ratio', 'mask_side_px']).agg(
    n_images=('val_index', 'nunique'),
    accuracy=('correct', 'mean'),
    target_margin_mean=('target_margin', 'mean'),
    target_prob_mean=('target_prob', 'mean'),
    target_margin_drop_mean=('target_margin_drop', 'mean'),
    target_prob_drop_mean=('target_prob_drop', 'mean'),
    top1_flip_rate=('top1_flip_from_clean', 'mean'),
    correct_flip_rate=('correct_flip_from_clean', 'mean'),
).reset_index()
summary.to_csv(OUT / 'black_mask_summary_by_area.csv', index=False)

display(summary.style.format({
    'mask_area_fraction': '{:.2f}',
    'mask_side_ratio': '{:.2f}',
    'accuracy': '{:.3f}',
    'target_margin_mean': '{:.3f}',
    'target_prob_mean': '{:.3f}',
    'target_margin_drop_mean': '{:.3f}',
    'target_prob_drop_mean': '{:.3f}',
    'top1_flip_rate': '{:.3f}',
    'correct_flip_rate': '{:.3f}',
}))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), constrained_layout=True)
for branch in BRANCHES:
    sub = summary[summary['branch_key'].eq(branch)].sort_values('mask_area_fraction')
    x = sub['mask_area_fraction']
    axes[0].plot(x, sub['accuracy'], marker='o', color=COLORS[branch], label=BRANCH_LABELS[branch])
    axes[1].plot(x, sub['target_margin_mean'], marker='o', color=COLORS[branch], label=BRANCH_LABELS[branch])
    axes[2].plot(x, sub['top1_flip_rate'], marker='o', color=COLORS[branch], label=BRANCH_LABELS[branch])
axes[0].set_title('Accuracy vs black-mask area')
axes[0].set_ylabel('accuracy')
axes[0].set_ylim(-0.02, 1.02)
axes[1].set_title('True-class margin vs black-mask area')
axes[1].set_ylabel('target logit - max other logit')
axes[2].set_title('Top-1 flip rate from clean prediction')
axes[2].set_ylabel('flip rate')
axes[2].set_ylim(-0.02, 1.02)
for ax in axes:
    ax.set_xlabel('black mask area fraction')
    ax.grid(alpha=0.25)
    ax.legend(frameon=False, fontsize=8)
fig.savefig(OUT / 'black_mask_summary_curves.png', dpi=180)
plt.show()

,branch_key,branch,mask_area_fraction,mask_side_ratio,mask_side_px,n_images,accuracy,target_margin_mean,target_prob_mean,target_margin_drop_mean,target_prob_drop_mean,top1_flip_rate,correct_flip_rate
0,logits,Strict AOG v39 skeleton primary relations,0.00,0.00,0,22,1.000,5.192,0.966,0.000,0.000,0.000,0.000
1,logits,Strict AOG v39 skeleton primary relations,0.36,0.60,134,22,0.682,1.293,0.496,3.899,0.471,0.318,0.318
2,logits,Strict AOG v39 skeleton primary relations,0.49,0.70,157,22,0.636,1.088,0.441,4.105,0.525,0.364,0.364
3,logits,Strict AOG v39 skeleton primary relations,0.64,0.80,179,22,0.273,-0.972,0.150,6.165,0.817,0.727,0.727
4,resnet_pretrained,ResNet-50 pretrained+FT,0.00,0.00,0,22,1.000,5.004,0.949,0.000,0.000,0.000,0.000
5,resnet_pretrained,ResNet-50 pretrained+FT,0.36,0.60,134,22,1.000,5.015,0.893,-0.011,0.056,0.000,0.000
6,resnet_pretrained,ResNet-50 pretrained+FT,0.49,0.70,157,22,0.909,4.128,0.840,0.876,0.109,0.091,0.091
7,resnet_pretrained,ResNet-50 pretrained+FT,0.64,0.80,179,22,0.455,0.007,0.307,4.996,0.642,0.545,0.545
8,resnet_scratch,ResNet-50 scratch,0.00,0.00,0,22,1.000,4.669,0.969,0.000,0.000,0.000,0.000
9,resnet_scratch,ResNet-50 scratch,0.36,0.60,134,22,0.636,0.550,0.439,4.119,0.530,0.364,0.364


/tmp/codex-tmp/ipykernel_5069/941408211.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Strict AOG Recognized-Part Overlays

This visualization runs the Strict AOG terminal extractor on the clean and black-masked images, then overlays the recognized part terminals. Colors indicate part type and labels show `part:score`. This is meant to show the structure our model still sees after each occlusion level.


In [7]:
STRUCTURE_TOP_TERMINALS = 12
STRUCTURE_MIN_SCORE = 0.02
PART_COLORS = list(plt.cm.tab20(np.linspace(0, 1, 20)))


def terminal_part_overlays_for_image(img: Image.Image) -> tuple[dict, dict]:
    stage_img = apply_stage_transform(img).unsqueeze(0).to(device, non_blocking=True)
    with torch.no_grad():
        stage1_out = stage1(stage_img)
        terminals = batch_extract_terminals(stage1_out, cfg=TERMINAL_CFG)
        out = strict_model(terminals, enable_edges=True)
    return {k: v.detach().cpu() if torch.is_tensor(v) else v for k, v in terminals.items()}, {k: v.detach().cpu() if torch.is_tensor(v) else v for k, v in out.items() if torch.is_tensor(v)}


def terminal_rows(terminals: dict) -> pd.DataFrame:
    valid = terminals['terminal_valid'][0].bool().numpy()
    parts = terminals['terminal_part'][0].long().numpy()
    scores = terminals['terminal_score'][0].float().numpy()
    support = terminals.get('terminal_support_overlap', torch.zeros_like(terminals['terminal_score']))[0].float().numpy()
    geom = terminals['terminal_geom'][0].float().numpy()
    masks = terminals['terminal_mask'][0].float().numpy()
    rows = []
    for tidx, ok in enumerate(valid):
        if not ok or float(scores[tidx]) < STRUCTURE_MIN_SCORE:
            continue
        part_id = int(parts[tidx])
        rows.append({
            'terminal_index': int(tidx),
            'part_id': part_id,
            'part_name': train_ds.schema.part_names[part_id] if 0 <= part_id < len(train_ds.schema.part_names) else str(part_id),
            'score': float(scores[tidx]),
            'support_overlap': float(support[tidx]),
            'cx': float(geom[tidx, 0]),
            'cy': float(geom[tidx, 1]),
            'w': float(geom[tidx, 2]),
            'h': float(geom[tidx, 3]),
            'area': float(geom[tidx, 4]),
            'mask': masks[tidx],
        })
    rows.sort(key=lambda r: (r['score'] * (0.5 + 0.5 * r['support_overlap']), r['area']), reverse=True)
    return pd.DataFrame(rows[:STRUCTURE_TOP_TERMINALS])


def overlay_terminals(ax, img: Image.Image, rows: pd.DataFrame, *, title: str):
    ax.imshow(img)
    ax.set_axis_off()
    ax.set_title(title, fontsize=10)
    if rows.empty:
        ax.text(0.5, 0.5, 'no terminals', transform=ax.transAxes, ha='center', va='center', fontsize=9, color='white', bbox=dict(facecolor='black', alpha=0.6, edgecolor='none'))
        return
    for _, r in rows.iterrows():
        part_id = int(r['part_id'])
        color = PART_COLORS[part_id % len(PART_COLORS)]
        mask = Image.fromarray((np.asarray(r['mask']) > 0.5).astype(np.uint8) * 255).resize((ANALYSIS_IMAGE_SIZE, ANALYSIS_IMAGE_SIZE), Image.Resampling.NEAREST)
        mask_arr = np.asarray(mask, dtype=float) / 255.0
        rgba = np.zeros((ANALYSIS_IMAGE_SIZE, ANALYSIS_IMAGE_SIZE, 4), dtype=float)
        rgba[..., :3] = color[:3]
        rgba[..., 3] = mask_arr * 0.36
        ax.imshow(rgba)
        cx = float(r['cx']) * (ANALYSIS_IMAGE_SIZE - 1)
        cy = float(r['cy']) * (ANALYSIS_IMAGE_SIZE - 1)
        ax.scatter([cx], [cy], s=18, c=[color[:3]], edgecolors='white', linewidths=0.5)
        ax.text(
            cx,
            cy,
            f"{r['part_name']}:{r['score']:.2f}",
            fontsize=6.5,
            color='white',
            ha='center',
            va='center',
            bbox=dict(facecolor='black', alpha=0.55, edgecolor='none', pad=1.0),
        )


def collect_structure_rows(selected_rows: pd.DataFrame) -> pd.DataFrame:
    records = []
    for _, row in tqdm(list(selected_rows.iterrows()), total=len(selected_rows), desc='part overlays'):
        clean = original_canvas(row['image_path'])
        variants = [(0.0, clean)] + [(float(area), add_black_mask(clean, float(area))[0]) for area in MASK_AREA_FRACTIONS]
        for area, img in variants:
            terminals, logits_out = terminal_part_overlays_for_image(img)
            rows = terminal_rows(terminals)
            pred = int(logits_out['logits'][0].argmax().item()) if 'logits' in logits_out else -1
            for _, tr in rows.drop(columns=['mask']).iterrows():
                records.append({
                    'val_index': int(row['val_index']),
                    'target_name': row['target_name'],
                    'mask_area_fraction': float(area),
                    'strict_pred': pred,
                    'strict_pred_name': class_names[pred] if 0 <= pred < len(class_names) else str(pred),
                    **tr.to_dict(),
                })
    return pd.DataFrame(records)


def show_part_overlay_page(rows: pd.DataFrame, page_name: str, *, max_rows: int = 6) -> None:
    rows = rows.head(max_rows)
    areas = [0.0] + MASK_AREA_FRACTIONS
    fig, axes = plt.subplots(len(rows), len(areas), figsize=(3.2 * len(areas), 2.85 * len(rows)), constrained_layout=True)
    if len(rows) == 1:
        axes = np.array([axes])
    for r, (_, row) in enumerate(rows.iterrows()):
        clean = original_canvas(row['image_path'])
        for c, area in enumerate(areas):
            img = clean if float(area) == 0.0 else add_black_mask(clean, float(area))[0]
            terminals, logits_out = terminal_part_overlays_for_image(img)
            trows = terminal_rows(terminals)
            pred = int(logits_out['logits'][0].argmax().item()) if 'logits' in logits_out else -1
            title = ('clean' if float(area) == 0.0 else f'area={area:.2f}') + f"\npred={class_names[pred] if 0 <= pred < len(class_names) else pred}"
            overlay_terminals(axes[r, c], img, trows, title=title)
            if c == 0:
                axes[r, c].set_ylabel(f"val {int(row['val_index'])}\n{row['target_name']}", fontsize=9)
    fig.savefig(OUT / page_name, dpi=180)
    plt.show()

# Save a compact CSV of recognized terminal parts and three pages of overlays.
structure_rows = collect_structure_rows(selected)
structure_rows.to_csv(OUT / 'strict_aog_recognized_parts_by_black_mask.csv', index=False)
show_part_overlay_page(selected, 'strict_aog_part_overlays_page1.png', max_rows=6)
if len(selected) > 6:
    show_part_overlay_page(selected.iloc[6:], 'strict_aog_part_overlays_page2.png', max_rows=6)
if len(selected) > 12:
    show_part_overlay_page(selected.iloc[12:], 'strict_aog_part_overlays_page3.png', max_rows=6)
if len(selected) > 18:
    show_part_overlay_page(selected.iloc[18:], 'strict_aog_part_overlays_page4.png', max_rows=6)

display(structure_rows.head(30))


part overlays:   0%|          | 0/22 [00:00<?, ?it/s]

part overlays:   5%|▍         | 1/22 [00:02<00:48,  2.33s/it]

part overlays:   9%|▉         | 2/22 [00:03<00:35,  1.77s/it]

part overlays:  14%|█▎        | 3/22 [00:08<00:56,  2.97s/it]

part overlays:  18%|█▊        | 4/22 [00:14<01:16,  4.24s/it]

...

part overlays:  95%|█████████▌| 21/22 [01:08<00:02,  2.89s/it]

part overlays: 100%|██████████| 22/22 [01:13<00:00,  3.71s/it]

part overlays: 100%|██████████| 22/22 [01:13<00:00,  3.35s/it]

/tmp/codex-tmp/ipykernel_5069/2226110869.py:114: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,val_index,target_name,mask_area_fraction,strict_pred,strict_pred_name,terminal_index,part_id,part_name,score,support_overlap,cx,cy,w,h,area
0,1159,aeroplane,0.00,0,aeroplane,2,5,head,0.954904,0.969724,0.448297,0.449815,0.165625,0.128125,0.014238
1,1159,aeroplane,0.00,0,aeroplane,1,0,body,0.923985,0.983822,0.365701,0.522003,0.340625,0.243750,0.047070
2,1159,aeroplane,0.00,0,aeroplane,4,1,engine,0.926786,0.963721,0.725335,0.558129,0.106250,0.068750,0.004756
3,1159,aeroplane,0.00,0,aeroplane,5,1,engine,0.901869,0.997854,0.613297,0.550104,0.056250,0.050000,0.002344
4,1159,aeroplane,0.00,0,aeroplane,0,12,wing,0.944426,0.853826,0.391785,0.554989,0.850000,0.193750,0.078242
5,1159,aeroplane,0.00,0,aeroplane,6,1,engine,0.809793,0.994979,0.175549,0.539595,0.040625,0.034375,0.001045
6,1159,aeroplane,0.00,0,aeroplane,3,10,tail,0.854199,0.861635,0.210268,0.472236,0.184375,0.184375,0.011631
7,1159,aeroplane,0.36,5,bottle,0,0,body,0.958447,0.954366,0.467023,0.566120,0.796875,0.503125,0.260801
8,1159,aeroplane,0.36,5,bottle,1,12,wing,0.902810,0.958535,0.106802,0.557137,0.171875,0.103125,0.014541
9,1159,aeroplane,0.36,5,bottle,2,5,head,0.713651,0.990277,0.608044,0.790659,0.159375,0.009375,0.001152


## Masked Image Examples

In [8]:
def show_mask_examples(rows: pd.DataFrame, page_name: str, *, max_rows: int = 8) -> None:
    rows = rows.head(max_rows)
    areas = [0.0] + MASK_AREA_FRACTIONS
    nrows = len(rows)
    ncols = len(areas)
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.0 * ncols, 2.7 * nrows), constrained_layout=True)
    if nrows == 1:
        axes = np.array([axes])
    for r, (_, row) in enumerate(rows.iterrows()):
        clean = original_canvas(row['image_path'])
        for c, area in enumerate(areas):
            img = clean if float(area) == 0.0 else add_black_mask(clean, float(area))[0]
            axes[r, c].imshow(img)
            axes[r, c].set_axis_off()
            if r == 0:
                axes[r, c].set_title('clean' if area == 0.0 else f'area={area:.2f}', fontsize=10)
            sub = results[(results['val_index'].eq(int(row['val_index']))) & (results['mask_area_fraction'].eq(float(area)))]
            label_parts = []
            for branch in BRANCHES:
                rec = sub[sub['branch_key'].eq(branch)].iloc[0]
                short = {'logits': 'AOG', 'resnet_pretrained': 'R-pre', 'resnet_scratch': 'R-scr'}[branch]
                label_parts.append(f"{short}:{rec['pred_name']}" + ('' if rec['correct'] else '*'))
            axes[r, c].set_xlabel('\n'.join(label_parts), fontsize=8)
        axes[r, 0].set_ylabel(f"val {int(row['val_index'])}\n{row['target_name']}", fontsize=9)
    fig.savefig(OUT / page_name, dpi=180)
    plt.show()

show_mask_examples(selected, 'black_mask_examples_page1.png', max_rows=8)
if len(selected) > 8:
    show_mask_examples(selected.iloc[8:], 'black_mask_examples_page2.png', max_rows=8)
if len(selected) > 16:
    show_mask_examples(selected.iloc[16:], 'black_mask_examples_page3.png', max_rows=8)

artifacts = sorted(OUT.glob('*'))
display(pd.DataFrame({'artifact': [str(p) for p in artifacts]}))

/tmp/codex-tmp/ipykernel_5069/1472534909.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,artifact
0,/home/dfli/instance_slot_aog/experiments/aog_v...
1,/home/dfli/instance_slot_aog/experiments/aog_v...
2,/home/dfli/instance_slot_aog/experiments/aog_v...
3,/home/dfli/instance_slot_aog/experiments/aog_v...
4,/home/dfli/instance_slot_aog/experiments/aog_v...
5,/home/dfli/instance_slot_aog/experiments/aog_v...
6,/home/dfli/instance_slot_aog/experiments/aog_v...
7,/home/dfli/instance_slot_aog/experiments/aog_v...
8,/home/dfli/instance_slot_aog/experiments/aog_v...
9,/home/dfli/instance_slot_aog/experiments/aog_v...
